# Phase 3: Batch Validation - Pipeline Preprocessing Cassava

**Objectif:** Valider le pipeline amélioré sur 1,000 images stratifiées

**Attendu:**
- Taux de succès: ≥95%
- Primary pipeline: ~85%
- Fallback: ~10-15%
- Échecs: <5%

**Inputs:**
- `../data/unified_dataset_manifest.csv` (30,513 images)
- Pipeline amélioré du notebook 03

**Outputs:**
- Métriques de validation
- Images problématiques identifiées
- Rapport de performance par classe

## 1. Setup & Imports

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import time
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import cv2
import xml.etree.ElementTree as ET

# Torch imports
import torch
from torchvision.ops import box_convert
import supervision as sv

# Transformers for GroundingDINO
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

# SAM
from segment_anything import sam_model_registry, SamPredictor

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

✓ Imports successful
PyTorch version: 2.9.1+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 5090


## 2. Configuration

In [2]:
# Paths
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs' / 'batch_validation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Unified manifest
MANIFEST_PATH = DATA_DIR / 'unified_dataset_manifest.csv'

# GroundingDINO model from Hugging Face
GROUNDING_DINO_MODEL_ID = "IDEA-Research/grounding-dino-tiny"

# SAM model
SAM_MODEL_TYPE = "vit_h"
SAM_CHECKPOINT = BASE_DIR / 'models' / 'sam_vit_h_4b8939.pth'

# Processing params
TARGET_SIZE = (448, 448)
PADDING_RATIO = 0.1

# Detection params
TEXT_THRESHOLD = 0.25
PROMPTS = ["cassava leaf", "cassava plant leaf", "green leaf"]

# Class-specific adaptive thresholds
CLASS_THRESHOLDS = {
    0: {'box_threshold': 0.30, 'min_area_ratio': 0.025},  # CBB - standard
    1: {'box_threshold': 0.30, 'min_area_ratio': 0.025},  # CBSD - standard
    2: {'box_threshold': 0.25, 'min_area_ratio': 0.015},  # CGM - MORE PERMISSIVE
    3: {'box_threshold': 0.30, 'min_area_ratio': 0.025},  # CMD - standard
    4: {'box_threshold': 0.28, 'min_area_ratio': 0.020},  # Healthy - slightly permissive
}

# Fallback configuration
FALLBACK_ENABLED = True
FALLBACK_METHOD = 'green_semantic'
GREEN_PERCENTILE_THRESHOLD = 60

# Batch validation params
SAMPLES_PER_CLASS = 200  # 200 * 5 = 1,000 total

# Class names
CLASS_NAMES = {
    0: 'Cassava Bacterial Blight (CBB)',
    1: 'Cassava Brown Streak Disease (CBSD)',
    2: 'Cassava Green Mottle (CGM)',
    3: 'Cassava Mosaic Disease (CMD)',
    4: 'Healthy'
}

print("✓ Configuration loaded")
print(f"GroundingDINO model: {GROUNDING_DINO_MODEL_ID}")
print(f"SAM checkpoint: {SAM_CHECKPOINT}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Validation batch size: {SAMPLES_PER_CLASS} per class = {SAMPLES_PER_CLASS * 5} total")

✓ Configuration loaded
GroundingDINO model: IDEA-Research/grounding-dino-tiny
SAM checkpoint: /home/hounfodjidagba/learning/cassava/models/sam_vit_h_4b8939.pth
Output directory: /home/hounfodjidagba/learning/cassava/outputs/batch_validation
Validation batch size: 200 per class = 1000 total


## 3. Load Models

In [3]:
# Load GroundingDINO from Hugging Face
print("Loading GroundingDINO from Hugging Face...")

grounding_dino_processor = AutoProcessor.from_pretrained(GROUNDING_DINO_MODEL_ID)
grounding_dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GROUNDING_DINO_MODEL_ID)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
grounding_dino_model.to(device)
grounding_dino_model.eval()

print(f"✓ GroundingDINO loaded on {device}")

# Load SAM model
print("\nLoading SAM model...")

sam = sam_model_registry[SAM_MODEL_TYPE](checkpoint=str(SAM_CHECKPOINT))
sam.to(device)
sam_predictor = SamPredictor(sam)

print(f"✓ SAM loaded on {device}")

print("\n" + "="*70)
print("MODELS READY")
print("="*70)

Loading GroundingDINO from Hugging Face...


✓ GroundingDINO loaded on cuda

Loading SAM model...
✓ SAM loaded on cuda

MODELS READY


## 4. Load Unified Dataset Manifest

In [4]:
# Load unified manifest
print("Loading unified dataset manifest...")
manifest_df = pd.read_csv(MANIFEST_PATH)

print(f"✓ Manifest loaded: {len(manifest_df):,} images")
print(f"\nDistribution by class:")
class_counts = manifest_df['label'].value_counts().sort_index()
for label, count in class_counts.items():
    print(f"  {label} - {CLASS_NAMES[label]:45s}: {count:,} images")

print(f"\nImages with XML annotations: {manifest_df['has_bbox_annotation'].sum():,}")
print(f"\nSources: {manifest_df['source'].nunique()}")
print(manifest_df['source'].value_counts().to_string())

Loading unified dataset manifest...
✓ Manifest loaded: 30,513 images

Distribution by class:
  0 - Cassava Bacterial Blight (CBB)               : 1,087 images
  1 - Cassava Brown Streak Disease (CBSD)          : 5,186 images
  2 - Cassava Green Mottle (CGM)                   : 2,386 images
  3 - Cassava Mosaic Disease (CMD)                 : 18,158 images
  4 - Healthy                                      : 3,696 images

Images with XML annotations: 2,993

Sources: 10
source
main           21397
cbsd-1          1787
cmd_002         1227
cmd_003         1220
cmd_001         1211
cbsd-2          1210
cmd_004          942
healthy_001      597
healthy_002      522
cmd_005          400


## 5. Import Pipeline Functions

We'll copy the essential functions from notebook 03.

In [5]:
# Detection function using Hugging Face GroundingDINO
def detect_leaves_grounding_dino(image_path, prompts, box_threshold, text_threshold):
    """Detect leaves using GroundingDINO from Hugging Face."""
    from PIL import Image
    
    image = Image.open(image_path).convert("RGB")
    
    # Prepare inputs
    caption = ". ".join(prompts)
    inputs = grounding_dino_processor(images=image, text=caption, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Run model
    with torch.no_grad():
        outputs = grounding_dino_model(**inputs)
    
    # Post-process outputs
    results = grounding_dino_processor.post_process_grounded_object_detection(
        outputs,
        inputs['input_ids'],
        threshold=box_threshold,
        text_threshold=text_threshold,
        target_sizes=[image.size[::-1]]
    )[0]
    
    # Extract boxes, scores, labels
    boxes = results['boxes'].cpu().numpy()  # Already in xyxy format
    scores = results['scores'].cpu().numpy()
    labels = results['labels']
    
    # Convert image to array for return
    image_array = np.array(image)
    
    return boxes, scores, labels, image_array

# SAM segmentation
def segment_leaf_sam(image_array, box):
    """Segment leaf using SAM."""
    sam_predictor.set_image(image_array)
    masks, scores, _ = sam_predictor.predict(
        box=box,
        multimask_output=True
    )
    best_mask_idx = np.argmax(scores)
    return masks[best_mask_idx], scores[best_mask_idx]

# Validation
def validate_detection(box, mask, image_shape, min_area_ratio):
    """Validate detection quality."""
    h, w = image_shape[:2]
    total_pixels = h * w
    
    box_w = box[2] - box[0]
    box_h = box[3] - box[1]
    box_area = box_w * box_h
    box_area_ratio = box_area / total_pixels
    
    mask_area = np.sum(mask)
    mask_area_ratio = mask_area / total_pixels
    
    is_valid = box_area_ratio >= min_area_ratio
    
    metrics = {
        'box_area_ratio': box_area_ratio,
        'mask_area_ratio': mask_area_ratio,
        'box_width': box_w,
        'box_height': box_h
    }
    
    return is_valid, metrics

# Green dominance check
def check_green_dominance(image_array, mask, threshold=1.2):
    """Check if masked region is predominantly green."""
    if np.sum(mask) == 0:
        return False, 0.0
    
    masked_pixels = image_array[mask]
    R = masked_pixels[:, 0].astype(np.float32)
    G = masked_pixels[:, 1].astype(np.float32)
    B = masked_pixels[:, 2].astype(np.float32)
    
    green_ratio = np.mean(G / ((R + B) / 2 + 1e-6))
    is_green = green_ratio >= threshold
    
    return is_green, float(green_ratio)

# Crop and resize
def crop_and_resize_leaf(image_array, mask, box, target_size=(448, 448), padding_ratio=0.1):
    """Crop leaf region with padding and resize."""
    h, w = image_array.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in box]
    
    # Add padding
    pad_w = int((x2 - x1) * padding_ratio)
    pad_h = int((y2 - y1) * padding_ratio)
    
    x1_pad = max(0, x1 - pad_w)
    y1_pad = max(0, y1 - pad_h)
    x2_pad = min(w, x2 + pad_w)
    y2_pad = min(h, y2 + pad_h)
    
    cropped = image_array[y1_pad:y2_pad, x1_pad:x2_pad]
    cropped_mask = mask[y1_pad:y2_pad, x1_pad:x2_pad]
    
    resized = cv2.resize(cropped, target_size, interpolation=cv2.INTER_LINEAR)
    resized_mask = cv2.resize(cropped_mask.astype(np.uint8), target_size, interpolation=cv2.INTER_NEAREST)
    
    return resized, resized_mask

print("✓ Pipeline helper functions defined (Hugging Face compatible)")

✓ Pipeline helper functions defined (Hugging Face compatible)


In [6]:
# Fallback: Green channel segmentation
def green_channel_segmentation(image_array, percentile_threshold=60):
    """Segmentation sémantique utilisant dominance du canal vert."""
    R = image_array[:, :, 0].astype(np.float32)
    G = image_array[:, :, 1].astype(np.float32)
    B = image_array[:, :, 2].astype(np.float32)
    
    green_ratio = G / ((R + B) / 2 + 1e-6)
    threshold_value = max(np.percentile(green_ratio, percentile_threshold), 1.05)
    mask = (green_ratio > threshold_value).astype(np.uint8)
    
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    
    if num_labels <= 1:
        return None, None
    
    areas = stats[1:, cv2.CC_STAT_AREA]
    largest_idx = 1 + np.argmax(areas)
    
    final_mask = (labels == largest_idx).astype(np.uint8)
    
    x = stats[largest_idx, cv2.CC_STAT_LEFT]
    y = stats[largest_idx, cv2.CC_STAT_TOP]
    w = stats[largest_idx, cv2.CC_STAT_WIDTH]
    h = stats[largest_idx, cv2.CC_STAT_HEIGHT]
    bbox = np.array([x, y, x + w, y + h])
    
    return final_mask, bbox

def fallback_preprocessing(image_array, method='green_semantic',
                          target_size=(448, 448), padding_ratio=0.1):
    """Fallback preprocessing when primary pipeline fails."""
    if method == 'green_semantic':
        mask, bbox = green_channel_segmentation(image_array)
        
        if mask is None or bbox is None:
            method = 'resize'
        else:
            processed, _ = crop_and_resize_leaf(image_array, mask, bbox, target_size, padding_ratio)
            
            metadata = {
                'method': method,
                'bbox': bbox.tolist(),
                'mask_area_ratio': np.sum(mask) / (image_array.shape[0] * image_array.shape[1])
            }
            
            return processed, metadata
    
    if method == 'center_crop':
        h, w = image_array.shape[:2]
        size = min(h, w)
        y1 = (h - size) // 2
        x1 = (w - size) // 2
        cropped = image_array[y1:y1+size, x1:x1+size]
        processed = cv2.resize(cropped, target_size, interpolation=cv2.INTER_LINEAR)
    else:  # resize
        processed = cv2.resize(image_array, target_size, interpolation=cv2.INTER_LINEAR)
    
    metadata = {'method': method, 'bbox': None, 'mask_area_ratio': 1.0}
    return processed, metadata

print("✓ Fallback functions defined")

✓ Fallback functions defined


## 6. Enhanced Preprocessing Pipeline

In [7]:
def preprocess_single_image(image_path,
                           label=None,
                           prompts=PROMPTS,
                           box_threshold=None,
                           text_threshold=TEXT_THRESHOLD,
                           min_area_ratio=None,
                           target_size=TARGET_SIZE,
                           padding_ratio=PADDING_RATIO,
                           enable_fallback=FALLBACK_ENABLED,
                           fallback_method=FALLBACK_METHOD,
                           return_debug_info=False):
    """
    Complete preprocessing pipeline WITH adaptive thresholds and fallback.
    
    Returns:
        success: Boolean
        result: Dictionary with processed image and metadata
    """
    # Apply class-specific thresholds
    if label is not None and label in CLASS_THRESHOLDS:
        class_config = CLASS_THRESHOLDS[label]
        box_threshold = box_threshold or class_config['box_threshold']
        min_area_ratio = min_area_ratio or class_config['min_area_ratio']
    else:
        box_threshold = box_threshold or 0.30
        min_area_ratio = min_area_ratio or 0.025
    
    image = Image.open(image_path).convert("RGB")
    image_array = np.array(image)
    
    # Try PRIMARY pipeline
    try:
        boxes, confidences, labels_detected, _ = detect_leaves_grounding_dino(
            image_path, prompts, box_threshold, text_threshold
        )
        
        if len(boxes) == 0:
            raise ValueError("No leaves detected by GroundingDINO")
        
        best_idx = np.argmax(confidences)
        best_box = boxes[best_idx]
        best_confidence = float(confidences[best_idx])
        best_label = labels_detected[best_idx] if len(labels_detected) > 0 else "leaf"
        
        mask, sam_confidence = segment_leaf_sam(image_array, best_box)
        
        is_valid, validation_metrics = validate_detection(
            best_box, mask, image_array.shape, min_area_ratio
        )
        
        if not is_valid:
            raise ValueError(f"Detection failed validation: {validation_metrics}")
        
        is_green, green_ratio = check_green_dominance(image_array, mask)
        processed_image, processed_mask = crop_and_resize_leaf(
            image_array, mask, best_box, target_size, padding_ratio
        )
        
        result = {
            "processed_image": processed_image,
            "processed_mask": processed_mask,
            "original_size": image_array.shape[:2],
            "box": best_box.tolist(),
            "detection_confidence": best_confidence,
            "sam_confidence": float(sam_confidence),
            "green_ratio": green_ratio,
            "is_green": is_green,
            "detected_label": best_label,
            "metrics": validation_metrics,
            "image_path": str(image_path),
            "method": "primary",
            "used_fallback": False,
            "class_label": label,
            "box_threshold_used": box_threshold,
            "min_area_ratio_used": min_area_ratio
        }
        
        if return_debug_info:
            result["original_image"] = image_array
            result["full_mask"] = mask
        
        return True, result
    
    except Exception as primary_error:
        if not enable_fallback:
            return False, {
                "error": f"Primary failed: {str(primary_error)}",
                "image_path": str(image_path),
                "method": "none",
                "used_fallback": False
            }
        
        # Try FALLBACK
        try:
            processed_image, fallback_metadata = fallback_preprocessing(
                image_array, method=fallback_method, target_size=target_size, padding_ratio=padding_ratio
            )
            
            result = {
                "processed_image": processed_image,
                "processed_mask": None,
                "original_size": image_array.shape[:2],
                "box": fallback_metadata.get('bbox'),
                "detection_confidence": None,
                "sam_confidence": None,
                "green_ratio": None,
                "is_green": None,
                "detected_label": None,
                "metrics": {'box_area_ratio': fallback_metadata.get('mask_area_ratio', 1.0)},
                "image_path": str(image_path),
                "method": f"fallback_{fallback_method}",
                "used_fallback": True,
                "primary_error": str(primary_error),
                "fallback_metadata": fallback_metadata,
                "class_label": label,
                "box_threshold_used": box_threshold,
                "min_area_ratio_used": min_area_ratio
            }
            
            if return_debug_info:
                result["original_image"] = image_array
            
            return True, result
        
        except Exception as fallback_error:
            return False, {
                "error": f"Both failed - Primary: {str(primary_error)}, Fallback: {str(fallback_error)}",
                "image_path": str(image_path),
                "method": "none",
                "used_fallback": False,
                "primary_error": str(primary_error),
                "fallback_error": str(fallback_error)
            }

print("✓ Enhanced preprocessing pipeline defined")

✓ Enhanced preprocessing pipeline defined


## 7. Create Stratified Validation Batch

In [8]:
def create_stratified_batch(manifest_df, samples_per_class=200, seed=42):
    """
    Create stratified batch from unified manifest.
    Ensures balanced representation across classes.
    
    Args:
        manifest_df: Unified manifest DataFrame
        samples_per_class: Number of samples per class
        seed: Random seed for reproducibility
    
    Returns:
        List of sample dictionaries
    """
    np.random.seed(seed)
    batch_samples = []
    
    print("Creating stratified validation batch...")
    print(f"Target: {samples_per_class} samples per class\n")
    
    for label in sorted(manifest_df['label'].unique()):
        class_df = manifest_df[manifest_df['label'] == label]
        
        # Sample randomly
        if len(class_df) >= samples_per_class:
            sampled = class_df.sample(n=samples_per_class, random_state=seed)
        else:
            print(f"  ⚠ Class {label} has only {len(class_df)} images (< {samples_per_class})")
            sampled = class_df
        
        batch_samples.extend(sampled.to_dict('records'))
        
        print(f"  Class {label} ({CLASS_NAMES[label][:30]}): {len(sampled)} samples")
    
    print(f"\n✓ Total batch size: {len(batch_samples)} images")
    return batch_samples

# Create the batch
validation_batch = create_stratified_batch(manifest_df, SAMPLES_PER_CLASS)

# Save batch list for reproducibility
batch_manifest_path = OUTPUT_DIR / 'validation_batch_manifest.json'
with open(batch_manifest_path, 'w') as f:
    json.dump(validation_batch, f, indent=2)
print(f"\n✓ Batch manifest saved: {batch_manifest_path}")

Creating stratified validation batch...
Target: 200 samples per class

  Class 0 (Cassava Bacterial Blight (CBB)): 200 samples
  Class 1 (Cassava Brown Streak Disease (): 200 samples
  Class 2 (Cassava Green Mottle (CGM)): 200 samples
  Class 3 (Cassava Mosaic Disease (CMD)): 200 samples
  Class 4 (Healthy): 200 samples

✓ Total batch size: 1000 images

✓ Batch manifest saved: /home/hounfodjidagba/learning/cassava/outputs/batch_validation/validation_batch_manifest.json


## 8. Process Validation Batch

In [9]:
print("="*70)
print("PROCESSING VALIDATION BATCH")
print("="*70)
print(f"Total images: {len(validation_batch)}")
print(f"Using: {'GPU' if torch.cuda.is_available() else 'CPU'}\n")

validation_results = []
failed_images = []

for sample in tqdm(validation_batch, desc="Processing"):
    image_path = Path(sample['file_path'])
    label = sample['label']
    
    try:
        start_time = time.time()
        success, result = preprocess_single_image(
            image_path,
            label=label,
            enable_fallback=True,
            return_debug_info=False
        )
        elapsed = time.time() - start_time
        
        # Add sample metadata
        result['image_id'] = sample['image_id']
        result['disease'] = sample['disease']
        result['label'] = label
        result['source'] = sample['source']
        result['success'] = success
        result['processing_time'] = elapsed
        
        validation_results.append(result)
        
        if not success:
            failed_images.append({
                'image_id': sample['image_id'],
                'label': label,
                'error': result.get('error', 'Unknown'),
                'file_path': str(image_path)
            })
    
    except Exception as e:
        print(f"\n✗ Error processing {sample['image_id']}: {str(e)}")
        failed_images.append({
            'image_id': sample['image_id'],
            'label': label,
            'error': str(e),
            'file_path': str(image_path)
        })

# Quick summary
successful = [r for r in validation_results if r['success']]
success_rate = len(successful) / len(validation_results) * 100

print(f"\n" + "="*70)
print(f"BATCH PROCESSING COMPLETE")
print("="*70)
print(f"Total processed: {len(validation_results)}")
print(f"Successful: {len(successful)} ({success_rate:.1f}%)")
print(f"Failed: {len(validation_results) - len(successful)}")
print("="*70)

# Save results
results_path = OUTPUT_DIR / 'validation_results.json'
with open(results_path, 'w') as f:
    # Convert to JSON-serializable format
    results_to_save = []
    for r in validation_results:
        r_copy = r.copy()
        # Remove numpy arrays
        r_copy.pop('processed_image', None)
        r_copy.pop('processed_mask', None)
        r_copy.pop('original_image', None)
        r_copy.pop('full_mask', None)
        results_to_save.append(r_copy)
    # Helper to convert numpy types
    def numpy_to_python(obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.bool_):
            return bool(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return obj
    
    json.dump(results_to_save, f, indent=2, default=numpy_to_python)

print(f"\n✓ Results saved: {results_path}")

if failed_images:
    failed_path = OUTPUT_DIR / 'failed_images.json'
    with open(failed_path, 'w') as f:
        json.dump(failed_images, f, indent=2)
    print(f"✓ Failed images list saved: {failed_path}")

PROCESSING VALIDATION BATCH
Total images: 1000
Using: GPU



Processing:   0%|          | 0/1000 [00:00<?, ?it/s]


BATCH PROCESSING COMPLETE
Total processed: 1000
Successful: 1000 (100.0%)
Failed: 0

✓ Results saved: /home/hounfodjidagba/learning/cassava/outputs/batch_validation/validation_results.json


## 9. Analyze Results

In [10]:
# Separate by method
primary_success = [r for r in successful if not r.get('used_fallback', False)]
fallback_success = [r for r in successful if r.get('used_fallback', False)]
failed = [r for r in validation_results if not r['success']]

print("="*70)
print("DETAILED ANALYSIS")
print("="*70)

print(f"\n📊 Overall Performance:")
print(f"  Total images: {len(validation_results)}")
print(f"  Successful: {len(successful)} ({len(successful)/len(validation_results)*100:.1f}%)")
print(f"    • Primary pipeline: {len(primary_success)} ({len(primary_success)/len(validation_results)*100:.1f}%)")
print(f"    • Fallback pipeline: {len(fallback_success)} ({len(fallback_success)/len(validation_results)*100:.1f}%)")
print(f"  Failed: {len(failed)} ({len(failed)/len(validation_results)*100:.1f}%)")

# Per-class analysis
print(f"\n📈 Per-Class Performance:")
for label in sorted(set(r['label'] for r in validation_results)):
    class_results = [r for r in validation_results if r['label'] == label]
    class_success = [r for r in class_results if r['success']]
    class_primary = [r for r in class_success if not r.get('used_fallback', False)]
    class_fallback = [r for r in class_success if r.get('used_fallback', False)]
    
    success_rate = len(class_success) / len(class_results) * 100 if class_results else 0
    
    print(f"\n  Class {label} - {CLASS_NAMES[label]}:")
    print(f"    Total: {len(class_results)}")
    print(f"    Success: {len(class_success)}/{len(class_results)} ({success_rate:.1f}%)")
    print(f"      Primary: {len(class_primary)}")
    print(f"      Fallback: {len(class_fallback)}")
    print(f"    Failed: {len(class_results) - len(class_success)}")

# Primary pipeline quality metrics
if primary_success:
    print(f"\n🎯 Primary Pipeline Quality:")
    det_confs = [r['detection_confidence'] for r in primary_success]
    sam_confs = [r['sam_confidence'] for r in primary_success]
    green_ratios = [r['green_ratio'] for r in primary_success]
    
    print(f"  Detection confidence: {np.mean(det_confs):.3f} ± {np.std(det_confs):.3f}")
    print(f"  SAM confidence: {np.mean(sam_confs):.3f} ± {np.std(sam_confs):.3f}")
    print(f"  Green ratio: {np.mean(green_ratios):.2f} ± {np.std(green_ratios):.2f}")
    print(f"  Green leaves: {sum(1 for r in primary_success if r.get('is_green', False))}/{len(primary_success)}")

# Processing time
times = [r['processing_time'] for r in validation_results]
print(f"\n⏱️  Processing Time:")
print(f"  Mean: {np.mean(times)*1000:.0f} ms/image")
print(f"  Median: {np.median(times)*1000:.0f} ms")
print(f"  Range: {np.min(times)*1000:.0f} - {np.max(times)*1000:.0f} ms")

# Failure analysis
if failed:
    print(f"\n❌ Failure Analysis:")
    print(f"  Total failures: {len(failed)}")
    
    # By class
    print(f"  By class:")
    for label in sorted(set(r['label'] for r in failed)):
        count = sum(1 for r in failed if r['label'] == label)
        print(f"    Class {label}: {count}")

# Validation verdict
print(f"\n" + "="*70)
if success_rate >= 95:
    print("✅ VALIDATION PASSED: Success rate ≥95%")
    print("   Pipeline is ready for production processing!")
elif success_rate >= 90:
    print("⚠️  VALIDATION MARGINAL: Success rate 90-95%")
    print("   Consider reviewing failed cases before production.")
else:
    print("❌ VALIDATION FAILED: Success rate <90%")
    print("   Pipeline needs further optimization.")
print("="*70)

DETAILED ANALYSIS

📊 Overall Performance:
  Total images: 1000
  Successful: 1000 (100.0%)
    • Primary pipeline: 960 (96.0%)
    • Fallback pipeline: 40 (4.0%)
  Failed: 0 (0.0%)

📈 Per-Class Performance:

  Class 0 - Cassava Bacterial Blight (CBB):
    Total: 200
    Success: 200/200 (100.0%)
      Primary: 192
      Fallback: 8
    Failed: 0

  Class 1 - Cassava Brown Streak Disease (CBSD):
    Total: 200
    Success: 200/200 (100.0%)
      Primary: 194
      Fallback: 6
    Failed: 0

  Class 2 - Cassava Green Mottle (CGM):
    Total: 200
    Success: 200/200 (100.0%)
      Primary: 195
      Fallback: 5
    Failed: 0

  Class 3 - Cassava Mosaic Disease (CMD):
    Total: 200
    Success: 200/200 (100.0%)
      Primary: 185
      Fallback: 15
    Failed: 0

  Class 4 - Healthy:
    Total: 200
    Success: 200/200 (100.0%)
      Primary: 194
      Fallback: 6
    Failed: 0

🎯 Primary Pipeline Quality:
  Detection confidence: 0.396 ± 0.032
  SAM confidence: 0.991 ± 0.013
  Green rati

In [11]:
import json
with open(OUTPUT_DIR / 'validation_results.json', 'r') as f:
    results = json.load(f)

# Vérifier les erreurs primary
primary_errors = [r.get('primary_error') for r in results if r.get('used_fallback')]
print("Erreurs primary (échantillon):")
for error in primary_errors[:5]:
    print(f"  - {error}")


Erreurs primary (échantillon):
  - Detection failed validation: {'box_area_ratio': np.float32(0.024901824), 'mask_area_ratio': np.float64(0.00866875), 'box_width': np.float32(158.62015), 'box_height': np.float32(75.35535)}
  - Detection failed validation: {'box_area_ratio': np.float32(0.0055315513), 'mask_area_ratio': np.float64(0.00301875), 'box_width': np.float32(29.417725), 'box_height': np.float32(90.25662)}
  - Detection failed validation: {'box_area_ratio': np.float32(0.020912806), 'mask_area_ratio': np.float64(0.007202083333333333), 'box_width': np.float32(122.197174), 'box_height': np.float32(82.147125)}
  - Detection failed validation: {'box_area_ratio': np.float32(0.021575442), 'mask_area_ratio': np.float64(0.010370833333333333), 'box_width': np.float32(66.95288), 'box_height': np.float32(154.67911)}
  - Detection failed validation: {'box_area_ratio': np.float32(0.020072334), 'mask_area_ratio': np.float64(0.012304166666666666), 'box_width': np.float32(78.49051), 'box_height':